In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "fortum_challenge_data"
results_schema = "04_results"

prediction_tables = {
    "with_weather": "consumption_hourly_predictions_with_weather",
    "without_weather": "consumption_hourly_predictions_without_weather",
    "weather_null": "consumption_hourly_predictions_weather_null"
}

baseline_table = "consumption_hourly_baseline"
actual_table = "consumption_hourly_actuals"

df_baseline = (
    spark.table(f"{catalog}.{results_schema}.{baseline_table}")
    .select(
        F.expr("timestamp_utc + interval 7 days").alias("timestamp_utc"),
        F.col("group_id").cast("int").alias("group_id"),
        F.col("target_consumption").alias("baseline_consumption")
    )
)

df_actual = (
    spark.table(f"{catalog}.{results_schema}.{actual_table}")
    .select(
        "timestamp_utc",
        F.col("group_id").cast("int").alias("group_id"),
        F.col("target_consumption").alias("actual_consumption")
    )
)

all_metrics = []

for model_name, pred_table in prediction_tables.items():
    # Read tables
    df_pred = (
        spark.table(f"{catalog}.{results_schema}.{pred_table}")
        .select(
            "timestamp_utc",
            F.col("group_id").cast("int").alias("group_id"),
            F.col("target_consumption").alias("predicted_consumption")
        )
    )

    # Join
    df_eval = (
        df_pred
        .join(df_actual, on=["timestamp_utc", "group_id"], how="inner")
        .join(df_baseline, on=["timestamp_utc", "group_id"], how="inner")
    )

    # Row-level errors
    df_eval = (
        df_eval
        .withColumn("pred_error", F.col("predicted_consumption") - F.col("actual_consumption"))
        .withColumn("base_error", F.col("baseline_consumption") - F.col("actual_consumption"))
        .withColumn("pred_abs_error", F.abs(F.col("pred_error")))
        .withColumn("base_abs_error", F.abs(F.col("base_error")))
        .withColumn(
            "pred_ape_percent",
            F.when(
                F.col("actual_consumption") != 0,
                F.abs(F.col("pred_error") / F.col("actual_consumption")) * 100
            )
        )
        .withColumn(
            "base_ape_percent",
            F.when(
                F.col("actual_consumption") != 0,
                F.abs(F.col("base_error") / F.col("actual_consumption")) * 100
            )
        )
    )

    model_metrics = (
            df_eval
            .agg(
                F.avg("pred_abs_error").alias("prediction_MAE"),
                F.avg("base_abs_error").alias("baseline_MAE"),
                F.avg("pred_ape_percent").alias("prediction_MAPE_percent"),
                F.avg("base_ape_percent").alias("baseline_MAPE_percent")
            )
            .withColumn(
                "FVA_percent",
                100 * (
                    F.col("baseline_MAPE_percent") - F.col("prediction_MAPE_percent")
                ) / F.col("baseline_MAPE_percent")
            )
            .withColumn("model", F.lit(model_name))
            .select(
                "model",
                "prediction_MAE",
                "baseline_MAE",
                "prediction_MAPE_percent",
                "baseline_MAPE_percent",
                "FVA_percent"
            )
        )
    all_metrics.append(model_metrics)

# Union all model metric rows into one result table
df_all_metrics = all_metrics[0]
for df in all_metrics[1:]:
    df_all_metrics = df_all_metrics.unionByName(df)

df_all_metrics.orderBy(F.desc("FVA_percent")).show(truncate=False)